In [12]:
import pandas as pd
import numpy as np
from collections import defaultdict

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv('birdsong.csv')

# ── Raw observation data (before any transformation) ──────────────
#
# One row per species-county-month observation. The same species-month
# combination can appear many times across different counties, and the same
# county can appear multiple times within a single species-month group.

print("=" * 70)
print("Raw observation data (before transformation)")
print("=" * 70)
print(df[['common_name', 'date', 'season', 'county', 'bird_count']]
      .head(8)
      .to_string(index=False))
print(f"\nTotal rows: {len(df):,}  |  Unique species: {df['common_name'].nunique()}"
      f"  |  Unique counties: {df['county'].nunique()}")

# ── After transaction construction ────────────────────────────────
#
# Group by (date, common_name) and collect all counties into a deduplicated
# sorted set. Each row becomes one transaction: a species observed in a given
# month, and the set of counties where it appeared. Deduplication ensures a
# county contributes at most one item per transaction regardless of how many
# individual sighting records it has.

transactions_df = (
    df.groupby(['date', 'common_name'])['county']
    .apply(lambda x: sorted(set(x)))
    .reset_index()
    .rename(columns={'county': 'counties (items)'})
)

print("\n" + "=" * 70)
print("After transaction construction (Spring 2021-03, first 6 rows)")
print("=" * 70)
spring_tx = transactions_df[transactions_df['date'] == '2021-03'].head(6)
for _, row in spring_tx.iterrows():
    items = row['counties (items)']
    counties_str = '{' + ', '.join(items[:6]) + \
                   (f', ... +{len(items)-6} more' if len(items) > 6 else '') + '}'
    print(f"  ({row['date']}, {row['common_name'][:32]:<32})  ->  {counties_str}")
print(f"\nTotal transactions (all seasons): {len(transactions_df):,}")

# ── Apriori implementation ────────────────────────────────────────────────────

def apriori_support(transactions, min_support):
    """Returns support scores for all frequent 1-itemsets."""
    n = len(transactions)
    item_counts = defaultdict(int)
    for t in transactions:
        for item in set(t):
            item_counts[frozenset([item])] += 1
    return {k: v / n for k, v in item_counts.items() if v / n >= min_support}

# ── Run Apriori per season ────────────────────────────────────────────────────

seasons   = ['Winter', 'Spring', 'Summer', 'Autumn']
all_results = {}

for season in seasons:
    sdf = df[df['season'] == season]

    # Each transaction = one (date × species) combo -> list of counties
    transactions = (
        sdf.groupby(['date', 'common_name'])['county']
        .apply(list)
        .tolist()
    )

    freq        = apriori_support(transactions, min_support=0.25)
    sorted_freq = sorted(freq.items(), key=lambda x: -x[1])
    top         = [(list(k)[0], round(v * 100, 1)) for k, v in sorted_freq[:10]]
    all_results[season] = {'top': top, 'n': len(transactions), 'freq': freq}

# ── Frequent itemsets output (all four seasons) ──────────────────
#
# After running Apriori on the full transaction list, counties meeting the
# minimum support threshold are returned as frequent 1-itemsets. Support =
# number of transactions containing the county / total transactions.
# Shown for all four seasons.

print("\n" + "=" * 70)
print("Frequent itemsets output (all seasons, support >= 25%)")
print("=" * 70)

for season in seasons:
    r           = all_results[season]
    n           = r['n']
    sorted_freq = sorted(r['freq'].items(), key=lambda x: -x[1])

    print(f"\n  {season}  (total transactions: {n:,})")
    print(f"  {'County':<20}  {'Support':>8}  {'Freq. count':>12}  Interpretation")
    print(f"  {'-' * 72}")
    for itemset, support in sorted_freq:
        county = list(itemset)[0]
        count  = int(support * n)
        print(f"  {county:<20}  {support*100:>7.1f}%  {count:>12,}  "
              f"Present in {support*100:.1f}% of {season} species-month transactions")

# ── Results tables: hotspot counties with top 3 species per season ───────────
#
# For each hotspot county, the top 3 species by total bird_count within that
# season are computed from the raw data and appended to the results table.

season_labels = {
    'Winter': 'Winter (December \u2013 February)',
    'Spring': 'Spring (March \u2013 May)',
    'Summer': 'Summer (June \u2013 August)',
    'Autumn': 'Autumn (September \u2013 November)',
}

print("\n" + "=" * 70)
print("RESULTS: Bird hotspots per season with top 3 species")
print("=" * 70)

for season in seasons:
    r   = all_results[season]
    sdf = df[df['season'] == season]

    print(f"\n{season_labels[season]}  (n = {r['n']:,} transactions)")
    print(f"  {'Rank':<6}  {'County':<20}  {'Support':>8}  Top 3 species")
    print(f"  {'-' * 78}")

    for i, (county, sup) in enumerate(r['top'], 1):
        top3 = (
            sdf[sdf['county'] == county]
            .groupby('common_name')['bird_count']
            .sum()
            .sort_values(ascending=False)
            .head(3)
            .index
            .tolist()
        )
        species_str = ', '.join(top3)
        print(f"  #{i:<5}  {county:<20}  {sup:>7.1f}%  {species_str}")

Raw observation data (before transformation)
                   common_name    date season   county  bird_count
            Band-tailed Pigeon 2021-01 Winter Garfield         1.0
                  Pine Warbler 2021-01 Winter  Boulder         1.0
           White-winged Scoter 2021-01 Winter  Larimer         1.0
                Brown Thrasher 2021-01 Winter  Boulder         1.0
              Bonaparte's Gull 2021-01 Winter   Pueblo         2.0
American Three-toed Woodpecker 2021-01 Winter  Larimer         1.0
            Greater Roadrunner 2021-01 Winter   Pueblo         1.0
 Graylag x Swan Goose (hybrid) 2021-01 Winter   Denver         3.0

Total rows: 349,430  |  Unique species: 567  |  Unique counties: 64

After transaction construction (Spring 2021-03, first 6 rows)
  (2021-03, Acorn Woodpecker                )  ->  {La Plata}
  (2021-03, American Avocet                 )  ->  {Boulder, Chaffee, Crowley, Denver, Mesa, Otero, ... +2 more}
  (2021-03, American Barn Owl               )

In [11]:
import pandas as pd
import numpy as np
from collections import defaultdict

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv('birdsong.csv')

# ── Raw observation data (before any transformation) ──────────────
#
# One row per species-county-month observation. The same species-month
# combination can appear many times across different counties, and the same
# county can appear multiple times within a single species-month group.

print("=" * 70)
print("Raw observation data (before transformation)")
print("=" * 70)
print(df[['common_name', 'date', 'season', 'county', 'bird_count']]
      .head(8)
      .to_string(index=False))
print(f"\nTotal rows: {len(df):,}  |  Unique species: {df['common_name'].nunique()}"
      f"  |  Unique counties: {df['county'].nunique()}")

# ── After transaction construction ────────────────────────────────
#
# Group by (date, common_name) and collect all counties into a deduplicated
# sorted set. Each row becomes one transaction: a species observed in a given
# month, and the set of counties where it appeared. Deduplication ensures a
# county contributes at most one item per transaction regardless of how many
# individual sighting records it has.

transactions_df = (
    df.groupby(['date', 'common_name'])['county']
    .apply(lambda x: sorted(set(x)))
    .reset_index()
    .rename(columns={'county': 'counties (items)'})
)

print("\n" + "=" * 70)
print("After transaction construction (Spring 2021-03, first 6 rows)")
print("=" * 70)
spring_tx = transactions_df[transactions_df['date'] == '2021-03'].head(6)
for _, row in spring_tx.iterrows():
    items = row['counties (items)']
    counties_str = '{' + ', '.join(items[:6]) + \
                   (f', ... +{len(items)-6} more' if len(items) > 6 else '') + '}'
    print(f"  ({row['date']}, {row['common_name'][:32]:<32})  ->  {counties_str}")
print(f"\nTotal transactions (all seasons): {len(transactions_df):,}")

# ── FP-Tree data structures ───────────────────────────────────────────────────

class FPNode:
    def __init__(self, item, count=0, parent=None):
        self.item      = item
        self.count     = count
        self.parent    = parent
        self.children  = {}
        self.node_link = None          # horizontal link to next node with same item

class FPTree:
    def __init__(self):
        self.root         = FPNode(None)
        self.header_table = {}         # item -> [total_count, first_node]

    def insert_transaction(self, transaction, count=1):
        node = self.root
        for item in transaction:
            if item in node.children:
                node.children[item].count += count
            else:
                new_node = FPNode(item, count, parent=node)
                node.children[item] = new_node
                self.header_table.setdefault(item, [0, None])
                # Append to node-link chain
                if self.header_table[item][1] is None:
                    self.header_table[item][1] = new_node
                else:
                    cur = self.header_table[item][1]
                    while cur.node_link:
                        cur = cur.node_link
                    cur.node_link = new_node
            self.header_table[item][0] += count
            node = node.children[item]

# ── Tree builder ──────────────────────────────────────────────────────────────

def build_fp_tree(transactions, min_support_count):
    item_counts = defaultdict(int)
    for t in transactions:
        for item in set(t):          # deduplicate within each transaction
            item_counts[item] += 1

    frequent_items = {k: v for k, v in item_counts.items()
                      if v >= min_support_count}

    tree = FPTree()
    for t in transactions:
        filtered = sorted(
            list(set(item for item in t if item in frequent_items)),
            key=lambda x: -frequent_items[x]
        )
        if filtered:
            tree.insert_transaction(filtered)

    return tree, frequent_items

# ── Prefix-path extractor ─────────────────────────────────────────────────────

def get_prefix_paths(item, header_table):
    """Walk the node-link chain and collect (prefix_path, count) pairs."""
    paths = []
    node  = header_table[item][1]
    while node:
        path, parent = [], node.parent
        while parent.item is not None:
            path.append(parent.item)
            parent = parent.parent
        if path:
            paths.append((list(reversed(path)), node.count))
        node = node.node_link
    return paths

# ── Recursive FP-Growth miner ─────────────────────────────────────────────────
#
# Key implementation note: prefix paths are inserted into the conditional
# FP-tree using insert_transaction(filtered, count=cnt) rather than repeating
# each path cnt times. The latter inflates the transaction count n and distorts
# support values for lower-frequency counties. Using weighted insertion keeps
# support calculations consistent with Apriori.

def fp_growth(tree, min_support_count, prefix=frozenset()):
    results = {}
    for item in list(tree.header_table):
        support = tree.header_table[item][0]
        if support < min_support_count:
            continue

        new_itemset = prefix | frozenset([item])
        results[new_itemset] = support

        # Build conditional FP-tree from prefix paths
        prefix_paths = get_prefix_paths(item, tree.header_table)
        if not prefix_paths:
            continue

        # Count frequent items in the conditional pattern base
        cond_item_counts = defaultdict(int)
        for path, cnt in prefix_paths:
            for p_item in path:
                cond_item_counts[p_item] += cnt

        frequent_cond = {k: v for k, v in cond_item_counts.items()
                         if v >= min_support_count}

        # Insert each prefix path once with its count (not repeated)
        cond_tree = FPTree()
        for path, cnt in prefix_paths:
            filtered = sorted(
                [p for p in path if p in frequent_cond],
                key=lambda x: -frequent_cond[x]
            )
            if filtered:
                cond_tree.insert_transaction(filtered, count=cnt)

        if cond_tree.header_table:
            results.update(fp_growth(cond_tree, min_support_count, new_itemset))

    return results

# ── Run FP-Growth per season ──────────────────────────────────────────────────

seasons     = ['Winter', 'Spring', 'Summer', 'Autumn']
all_results = {}

for season in seasons:
    sdf = df[df['season'] == season]

    # Each transaction = one (date × species) combo -> list of counties
    transactions = (
        sdf.groupby(['date', 'common_name'])['county']
        .apply(list)
        .tolist()
    )

    n             = len(transactions)
    min_sup_count = int(0.25 * n)

    tree, _        = build_fp_tree(transactions, min_sup_count)
    freq_itemsets  = fp_growth(tree, min_sup_count)

    # Filter to single-county itemsets and sort by support descending
    singles     = {k: v for k, v in freq_itemsets.items() if len(k) == 1}
    sorted_freq = sorted(singles.items(), key=lambda x: -x[1])
    top         = [(list(k)[0], round(v / n * 100, 1)) for k, v in sorted_freq[:10]]

    all_results[season] = {'top': top, 'n': n, 'freq': singles,
                           'sorted_freq': sorted_freq}

# ── Frequent itemsets output (all four seasons) ──────────────────
#
# After FP-Growth mines the full transaction tree, counties meeting the minimum
# support threshold are returned as frequent 1-itemsets. Support = raw count
# from the FP-tree / total number of transactions in the season.
# Shown for all four seasons.

print("\n" + "=" * 70)
print("Frequent itemsets output (all seasons, support >= 25%)")
print("=" * 70)

for season in seasons:
    r           = all_results[season]
    n           = r['n']
    sorted_freq = r['sorted_freq']

    print(f"\n  {season}  (total transactions: {n:,})")
    print(f"  {'County':<20}  {'Support':>8}  {'Freq. count':>12}  Interpretation")
    print(f"  {'-' * 72}")
    for itemset, count in sorted_freq:
        county  = list(itemset)[0]
        support = count / n
        print(f"  {county:<20}  {support*100:>7.1f}%  {count:>12,}  "
              f"Present in {support*100:.1f}% of {season} species-month transactions")

# ── Results tables: hotspot counties with top 3 species per season ───────────
#
# For each hotspot county, the top 3 species by total bird_count within that
# season are computed from the raw data and appended to the results table.

season_labels = {
    'Winter': 'Winter (December \u2013 February)',
    'Spring': 'Spring (March \u2013 May)',
    'Summer': 'Summer (June \u2013 August)',
    'Autumn': 'Autumn (September \u2013 November)',
}

print("\n" + "=" * 70)
print("RESULTS: Bird hotspots per season with top 3 species")
print("=" * 70)

for season in seasons:
    r   = all_results[season]
    sdf = df[df['season'] == season]

    print(f"\n{season_labels[season]}  (n = {r['n']:,} transactions)")
    print(f"  {'Rank':<6}  {'County':<20}  {'Support':>8}  Top 3 species")
    print(f"  {'-' * 78}")

    for i, (county, sup) in enumerate(r['top'], 1):
        top3 = (
            sdf[sdf['county'] == county]
            .groupby('common_name')['bird_count']
            .sum()
            .sort_values(ascending=False)
            .head(3)
            .index
            .tolist()
        )
        species_str = ', '.join(top3)
        print(f"  #{i:<5}  {county:<20}  {sup:>7.1f}%  {species_str}")

Raw observation data (before transformation)
                   common_name    date season   county  bird_count
            Band-tailed Pigeon 2021-01 Winter Garfield         1.0
                  Pine Warbler 2021-01 Winter  Boulder         1.0
           White-winged Scoter 2021-01 Winter  Larimer         1.0
                Brown Thrasher 2021-01 Winter  Boulder         1.0
              Bonaparte's Gull 2021-01 Winter   Pueblo         2.0
American Three-toed Woodpecker 2021-01 Winter  Larimer         1.0
            Greater Roadrunner 2021-01 Winter   Pueblo         1.0
 Graylag x Swan Goose (hybrid) 2021-01 Winter   Denver         3.0

Total rows: 349,430  |  Unique species: 567  |  Unique counties: 64

After transaction construction (Spring 2021-03, first 6 rows)
  (2021-03, Acorn Woodpecker                )  ->  {La Plata}
  (2021-03, American Avocet                 )  ->  {Boulder, Chaffee, Crowley, Denver, Mesa, Otero, ... +2 more}
  (2021-03, American Barn Owl               )